In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
import copy
import pickle
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import (
    precision_score, recall_score, f1_score, roc_auc_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_curve
)

from sklearn.impute import SimpleImputer
from sklearn.linear_model    import LogisticRegression
from sklearn.tree            import DecisionTreeClassifier
from sklearn.ensemble        import RandomForestClassifier, AdaBoostClassifier, VotingClassifier
from sklearn.neighbors       import KNeighborsClassifier
from sklearn.svm             import SVC
import xgboost as xgb

from imblearn.over_sampling  import SMOTE, RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler

In [2]:
dt = pd.read_csv('../data/processed/train_cleaned.csv')

In [3]:
X = dt.drop(columns=['isFraud'])
y = dt['isFraud']

In [4]:
numeric_cols = X.select_dtypes(include=['int64', 'float64', 'int32', 'float32']).columns

print("Log: Imputing numeric missing values with median...")
numeric_imputer = SimpleImputer(strategy='median')

X[numeric_cols] = numeric_imputer.fit_transform(X[numeric_cols])

print(f"Log: Total NaNs remaining in X: {X.isnull().sum().sum()}")

Log: Imputing numeric missing values with median...
Log: Total NaNs remaining in X: 0


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,       # 80% train, 20% test
    random_state=42,    
    stratify=y           
)

print(f'Train: {X_train.shape[0]:,} rows  |  Fraud Rate: {y_train.mean():.3%}')
print(f'Test : {X_test.shape[0]:,} rows   |  Fraud Rate: {y_test.mean():.3%}')
print(f'\nFraud ratio maintained: {y.mean():.3%} → Train {y_train.mean():.3%} / Test {y_test.mean():.3%}')

Train: 472,432 rows  |  Fraud Rate: 3.499%
Test : 118,108 rows   |  Fraud Rate: 3.499%

Fraud ratio maintained: 3.499% → Train 3.499% / Test 3.499%


In [6]:
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)   # fit() ONLY on train
X_test_s  = scaler.transform(X_test)        # transform() applied to both

# Convert Numpy array back to DataFrame (to keep feature names)
X_train_s = pd.DataFrame(X_train_s, columns=X_train.columns)
X_test_s  = pd.DataFrame(X_test_s,  columns=X_test.columns)

print('Scaling complete.')
print(f'Train mean ≈ 0: {X_train_s.mean().mean():.6f}')
print(f'Train std  ≈ 1: {X_train_s.std().mean():.6f}')

Scaling complete.
Train mean ≈ 0: -0.000000
Train std  ≈ 1: 0.988143


In [7]:
missing_counts = X_train_s.isnull().sum()
columns_with_nans = missing_counts[missing_counts > 0]

print("Columns still containing NaNs:\n", columns_with_nans)

Columns still containing NaNs:
 Series([], dtype: int64)
